# Proyecto Integrador ? Fase 1: Predicci?n del precio de viviendas

**Curso:** Modelos y Simulaci?n de Sistemas I  \  
**Problema:** Regresi?n supervisada  \  
**Variable objetivo:** `price`

El objetivo es construir un modelo capaz de predecir el precio de venta de una vivienda a partir de sus caracter?sticas f?sicas, ubicaci?n y fecha de venta. El an?lisis incluye exploraci?n cuantitativa, preparaci?n reproducible, selecci?n interna de modelos y evaluaci?n final sobre una prueba reservada.

**Fuente exacta del dataset:** [House price prediction ? Kaggle](https://www.kaggle.com/datasets/shree1992/housedata). Es un conjunto de datos de viviendas vendidas en el ?rea de Seattle/Washington, con 4.600 registros y 18 columnas. Incluye precio, dimensiones habitables y del lote, habitaciones, ba?os, pisos, vista, condici?n, fecha y variables de ubicaci?n. `price_per_sqft` es una variable derivada del precio y se excluye de los predictores para evitar fuga de informaci?n.


**Entorno reproducible vigente:** Python 3.14.3, pandas 3.0.3, NumPy 2.4.6, scikit-learn 1.9.0 y joblib 1.5.3. Las versiones estan fijadas en `fase-1/requirements.txt`. El archivo `modelo.joblib` historico fue serializado con scikit-learn 1.6.1; scikit-learn no garantiza compatibilidad entre versiones para artefactos serializados. Para ese archivo se documenta un perfil separado en `requirements-modelo-legado.txt`; la opcion recomendada es regenerarlo en el entorno vigente.


## 1. Importación de librerías y carga del conjunto de datos

Para garantizar la reproducibilidad, se fija `random_state=42` en los procedimientos aleatorios. El archivo `DataSet_House_Price_Prediction.csv` debe estar ubicado junto al notebook dentro de la carpeta `fase-1`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import sklearn

from sklearn.model_selection import GroupShuffleSplit, GroupKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Versiones del entorno:")
print("Python: consulte sys.version en el entorno activo")
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("joblib:", joblib.__version__)


In [ ]:
# Buscar el CSV en ubicaciones relativas habituales.
rutas_posibles = [
    Path("DataSet_House_Price_Prediction.csv"),
    Path("fase-1") / "DataSet_House_Price_Prediction.csv"
]

ruta_csv = next((ruta for ruta in rutas_posibles if ruta.exists()), None)

if ruta_csv is None:
    raise FileNotFoundError(
        "No se encontró DataSet_House_Price_Prediction.csv. "
        "Ubique el CSV junto al notebook dentro de la carpeta fase-1."
    )

df_original = pd.read_csv(ruta_csv)
df = df_original.copy()

print("Archivo cargado:", ruta_csv)
print("Dimensiones originales:", df.shape)
display(df.head())

## 2. Descripci?n y an?lisis exploratorio

El conjunto contiene informaci?n f?sica, geogr?fica y temporal de viviendas. El EDA no se limita a producir gr?ficos: cada an?lisis se acompa?a de una interpretaci?n sobre distribuci?n, valores extremos, relaciones entre variables y consecuencias para el modelado.


In [ ]:
print("Dimensiones:", df.shape)
print("\nTipos de datos:")
display(df.dtypes.to_frame("tipo"))

print("\nValores faltantes originales por variable:")
faltantes_originales = df.isna().sum().sort_values(ascending=False)
display(faltantes_originales.to_frame("nulos"))
print("Total de faltantes originales:", int(faltantes_originales.sum()))

print("\nEstad?sticas descriptivas:")
display(df.describe(include="all").T)

print("\nContexto de la variable objetivo:")
print("Precio mediano:", round(df["price"].median(), 2))
print("Precio m?ximo:", round(df["price"].max(), 2))
print("Registros con price = 0:", int((df["price"] == 0).sum()))


In [ ]:
# Distribuciones de variables numéricas relevantes
variables_graficas = ["price", "sqft_living", "sqft_lot", "bedrooms", "bathrooms"]

for variable in variables_graficas:
    plt.figure(figsize=(7, 4))
    plt.hist(df[variable].dropna(), bins=30)
    plt.title(f"Distribución de {variable}")
    plt.xlabel(variable)
    plt.ylabel("Frecuencia")
    plt.show()

    plt.figure(figsize=(7, 2.5))
    plt.boxplot(df[variable].dropna(), vert=False)
    plt.title(f"Boxplot de {variable}")
    plt.xlabel(variable)
    plt.show()

In [ ]:
# Correlaci?n entre variables num?ricas
numericas_eda = df.select_dtypes(include=np.number)
correlaciones = numericas_eda.corr()

print("Correlaci?n de las variables num?ricas con price:")
display(
    correlaciones["price"]
    .drop("price")
    .sort_values(ascending=False)
    .to_frame("correlacion_con_price")
)

corr_living_above = correlaciones.loc["sqft_living", "sqft_above"]
print("\nCorrelaci?n sqft_living vs sqft_above:", round(corr_living_above, 4))
print(
    "Interpretaci?n: ambas variables miden superficies relacionadas, pero no id?nticas. "
    "sqft_living representa el ?rea habitable total y sqft_above la parte sobre el nivel del suelo; "
    "la diferencia aproxima el s?tano."
)

if corr_living_above >= 0.80:
    print(
        "Aunque existe alta correlaci?n, se conservan ambas porque el s?tano puede aportar informaci?n "
        "adicional sobre el precio y los modelos de ?rboles toleran variables correlacionadas. "
        "Como mejora futura se podr?a comparar contra una versi?n con una sola superficie o con "
        "la variable derivada sqft_basement/superficie sobre el nivel del suelo."
    )


### Hallazgos del EDA

- El precio presenta una distribuci?n asim?trica a la derecha: unas pocas viviendas de precio muy alto pueden influir fuertemente en RMSE. Por ello no se eliminan autom?ticamente los valores extremos; primero se verifica si representan ventas plausibles.
- `sqft_living` y `sqft_above` tienen una correlaci?n aproximada de 0,88. Son variables relacionadas, pero `sqft_living` incluye la superficie habitable total y `sqft_above` solo la superficie sobre el nivel del suelo. Se conservan porque la diferencia contiene informaci?n sobre s?tanos y el modelo basado en ?rboles no requiere eliminar variables correlacionadas. La decisi?n debe contrastarse posteriormente con validaci?n cruzada.
- `price_per_sqft` tiene relaci?n directa con `price`, porque se calcula usando la variable objetivo. Por eso se excluye antes del entrenamiento.
- Los 49 registros con `price = 0` no representan un precio de venta ?til y se eliminan. El dataset original no presenta valores faltantes; los faltantes del 1 % se simulan despu?s ?nicamente para comprobar el comportamiento de imputaci?n.


In [ ]:
print("Registros totales:", len(df))
print("Registros con price = 0:", (df["price"] == 0).sum())

print("\nCasos con mayor price_per_sqft:")
display(
    df[["price", "sqft_living", "price_per_sqft"]]
    .sort_values("price_per_sqft", ascending=False)
    .head(10)
)

## 3. Valores faltantes y limpieza

La inspecci?n confirma que el dataset original no presenta valores faltantes. Para evaluar de forma controlada la imputaci?n, se simula un 1 % de faltantes ?nicamente en `sqft_living`, usando una semilla fija. Esta simulaci?n no debe interpretarse como una caracter?stica original del dataset.

La imputaci?n se realiza posteriormente dentro de un `Pipeline`, despu?s de separar desarrollo y prueba. As?, la mediana se aprende solo con el conjunto de desarrollo y no se filtra informaci?n desde la prueba final.


In [ ]:
df = df_original.copy()

# Simulaci?n controlada: el dataset original no contiene faltantes.
np.random.seed(RANDOM_STATE)
cantidad_nulos = int(len(df) * 0.01)
filas_nulas = np.random.choice(df.index, cantidad_nulos, replace=False)
df.loc[filas_nulas, "sqft_living"] = np.nan

print("Faltantes originales antes de simular:", int(df_original.isna().sum().sum()))
print("Valores faltantes simulados en sqft_living:", int(df["sqft_living"].isna().sum()))
print("Porcentaje simulado:", round(df["sqft_living"].isna().mean() * 100, 2), "%")

# Eliminar registros con precio igual a cero
df = df[df["price"] > 0].copy()

print("\nDimensiones despu?s de retirar price = 0:", df.shape)
print("Valores faltantes simulados restantes en sqft_living:", int(df["sqft_living"].isna().sum()))


## 4. Preparación de variables

`price` es la variable objetivo. `price_per_sqft` se elimina para evitar fuga de información. La fecha se transforma en año y mes de venta. `street` se elimina por su cardinalidad extremadamente alta, mientras que `city` y `statezip` se conservan y se codifican mediante One-Hot Encoding.

In [ ]:
y = df["price"]
X = df.drop(columns=["price", "price_per_sqft"]).copy()

# Transformar la fecha
X["date"] = pd.to_datetime(X["date"])
X["year_sale"] = X["date"].dt.year
X["month_sale"] = X["date"].dt.month
X = X.drop(columns=["date"])

print("Cardinalidad de variables de ubicación:")
for columna in ["street", "city", "statezip"]:
    print(f"{columna}: {X[columna].nunique()} categorías")

# street presenta casi una categoría por vivienda
X = X.drop(columns=["street"])

print("\nNúmero final de variables predictoras:", X.shape[1])
print(X.columns.tolist())

## 5. Separaci?n de desarrollo y prueba final

La prueba final se reserva desde este punto y no se utiliza para comparar algoritmos ni ajustar hiperpar?metros. Como el dataset contiene viviendas repetidas por combinaci?n de direcci?n y ciudad, se utiliza esa combinaci?n como grupo: ninguna direcci?n-ciudad aparece simult?neamente en desarrollo y prueba.


In [ ]:
# Grupo de evaluaci?n: viviendas con la misma direcci?n y ciudad
# Se conserva street solo para definir grupos; no se usa como predictor.
grupos = (
    df["street"].fillna("<sin_direccion>").astype(str).str.strip()
    + "|"
    + df["city"].fillna("<sin_ciudad>").astype(str).str.strip()
)

separador = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)
indices_desarrollo, indices_prueba = next(separador.split(df, y, groups=grupos))

X_desarrollo = X.iloc[indices_desarrollo].copy()
X_prueba = X.iloc[indices_prueba].copy()
y_desarrollo = y.iloc[indices_desarrollo].copy()
y_prueba = y.iloc[indices_prueba].copy()
grupos_desarrollo = grupos.iloc[indices_desarrollo]
grupos_prueba = grupos.iloc[indices_prueba]

# Verificaci?n expl?cita: no debe haber grupos compartidos entre desarrollo y prueba.
grupos_compartidos = set(grupos_desarrollo).intersection(set(grupos_prueba))
print("Desarrollo:", X_desarrollo.shape)
print("Prueba final:", X_prueba.shape)
print("Grupos compartidos entre desarrollo y prueba:", len(grupos_compartidos))
assert not grupos_compartidos, "La separaci?n por grupos dej? direcciones compartidas."


## 6. An?lisis de viviendas repetidas y prevenci?n de fuga de informaci?n

La partici?n aleatoria original permit?a que una misma combinaci?n `street`-`city` apareciera en entrenamiento y prueba; en la revisi?n se observaron 25 combinaciones compartidas. Esto no demuestra por s? solo fuga de informaci?n, porque `street` se excluye de los predictores y las transformaciones se ajustan solo con desarrollo. Sin embargo, puede hacer que la evaluaci?n represente mejor el caso de predecir viviendas de direcciones ya observadas que el caso de generalizar a una direcci?n nueva.

Por ese motivo, la evaluaci?n principal usa `GroupShuffleSplit` con `street` y `city` como clave de grupo. As? se evita compartir una vivienda/direcci?n entre desarrollo y prueba. La selecci?n de modelos tambi?n usa `GroupKFold` dentro de desarrollo, manteniendo la misma separaci?n por grupos. La prueba final permanece intacta hasta la evaluaci?n final.


In [ ]:
categoricas = ["city", "statezip"]
numericas = [col for col in X_desarrollo.columns if col not in categoricas]

pipeline_numerico = Pipeline(steps=[
    ("imputacion", SimpleImputer(strategy="median"))
])

preprocesador = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categoricas),
        ("num", pipeline_numerico, numericas)
    ]
)

print("Variables categ?ricas:", len(categoricas))
print("Variables num?ricas:", len(numericas))


## 7. Selecci?n interna de modelos

La selecci?n se realiza ?nicamente sobre el conjunto de desarrollo mediante validaci?n cruzada por grupos. El conjunto de prueba final no participa en esta comparaci?n. Se utiliza MAE como criterio principal y RMSE y R? como m?tricas complementarias.


In [ ]:
modelos_candidatos = {
    "Regresi?n Lineal": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        random_state=RANDOM_STATE
    )
}

cv_grupos = GroupKFold(n_splits=5)
resultados_cv = []
pipelines_candidatos = {}

for nombre, estimador in modelos_candidatos.items():
    pipeline = Pipeline(steps=[
        ("preprocesamiento", preprocesador),
        ("modelo", estimador)
    ])
    scores = cross_validate(
        pipeline,
        X_desarrollo,
        y_desarrollo,
        groups=grupos_desarrollo,
        cv=cv_grupos,
        scoring={
            "MAE": "neg_mean_absolute_error",
            "RMSE": "neg_root_mean_squared_error",
            "R2": "r2"
        },
        n_jobs=-1
    )
    resultados_cv.append({
        "Modelo": nombre,
        "MAE_promedio": -scores["test_MAE"].mean(),
        "MAE_std": scores["test_MAE"].std(),
        "RMSE_promedio": -scores["test_RMSE"].mean(),
        "R2_promedio": scores["test_R2"].mean()
    })
    pipelines_candidatos[nombre] = pipeline

resultados_cv = pd.DataFrame(resultados_cv).sort_values("MAE_promedio")
display(resultados_cv.round(2))

nombre_modelo_seleccionado = resultados_cv.iloc[0]["Modelo"]
modelo_final = pipelines_candidatos[nombre_modelo_seleccionado]
print("Modelo seleccionado por MAE promedio de validaci?n:", nombre_modelo_seleccionado)


## 8. Evaluaci?n final sobre la prueba reservada

Despu?s de seleccionar el modelo con validaci?n cruzada por grupos, se entrena una sola vez con todo el conjunto de desarrollo. Solo entonces se eval?a sobre la prueba final, que no fue utilizada durante la selecci?n.


In [ ]:
modelo_final.fit(X_desarrollo, y_desarrollo)
y_pred_final = modelo_final.predict(X_prueba)

mae_final = mean_absolute_error(y_prueba, y_pred_final)
rmse_final = np.sqrt(mean_squared_error(y_prueba, y_pred_final))
r2_final = r2_score(y_prueba, y_pred_final)

print("MODELO FINAL -", nombre_modelo_seleccionado)
print("MAE:", round(mae_final, 2))
print("RMSE:", round(rmse_final, 2))
print("R?:", round(r2_final, 4))


## 9. An?lisis de errores y evaluaci?n final

El modelo seleccionado se entrena con todo el conjunto de desarrollo y se eval?a una sola vez sobre la prueba final. Adem?s de MAE, RMSE y R?, se inspeccionan los errores individuales para identificar qu? observaciones explican la penalizaci?n elevada de RMSE.


In [ ]:
errores = pd.DataFrame({
    "price_real": y_prueba.to_numpy(),
    "price_predicho": y_pred_final,
})
errores["error"] = errores["price_real"] - errores["price_predicho"]
errores["error_absoluto"] = errores["error"].abs()
errores["error_cuadratico"] = errores["error"] ** 2

print("\nDiagn?stico de errores grandes:")
print("Error absoluto m?ximo:", round(errores["error_absoluto"].max(), 2))
print("Mediana del error absoluto:", round(errores["error_absoluto"].median(), 2))
print(
    "RMSE / MAE:",
    round(rmse_final / mae_final, 2),
    "-> valores altos indican que algunos errores grandes pesan considerablemente m?s que el error t?pico."
)

for porcentaje in [1, 5, 10]:
    cantidad = max(1, int(np.ceil(len(errores) * porcentaje / 100)))
    contribucion = (
        errores.nlargest(cantidad, "error_cuadratico")["error_cuadratico"].sum()
        / errores["error_cuadratico"].sum()
        * 100
    )
    print(f"El {porcentaje}% de los casos con mayor error aporta {contribucion:.2f}% del SSE total.")

print("\nDiez errores absolutos m?s grandes:")
display(errores.nlargest(10, "error_absoluto").round(2))

# Comparaci?n por grupos de precio para detectar si el desempe?o cambia con la escala.
errores["grupo_precio"] = pd.qcut(
    errores["price_real"], q=4, duplicates="drop"
)
resumen_errores = errores.groupby("grupo_precio", observed=True).apply(
    lambda grupo: pd.Series({
        "n": len(grupo),
        "MAE": mean_absolute_error(grupo["price_real"], grupo["price_predicho"]),
        "RMSE": np.sqrt(mean_squared_error(grupo["price_real"], grupo["price_predicho"])),
        "error_absoluto_max": grupo["error_absoluto"].max()
    })
)
print("\nErrores por cuartil del precio real:")
display(resumen_errores.round(2))

plt.figure(figsize=(7, 5))
plt.scatter(errores["price_real"], errores["price_predicho"], alpha=0.45)
limite = max(errores["price_real"].max(), errores["price_predicho"].max())
plt.plot([0, limite], [0, limite], "--", color="red")
plt.xlabel("Precio real")
plt.ylabel("Precio predicho")
plt.title("Precio real vs. precio predicho")
plt.show()

print(
    "Interpretaci?n: RMSE es mayor que MAE porque eleva al cuadrado los errores. "
    "Las tablas anteriores muestran si la diferencia proviene de pocas viviendas extremas "
    "y si los errores se concentran en los precios m?s altos."
)


In [ ]:
# Comparaci?n contra un baseline constante, calculado tambi?n sin tocar la prueba final
modelo_base = DummyRegressor(strategy="median")
base_cv = cross_validate(
    modelo_base,
    X_desarrollo,
    y_desarrollo,
    groups=grupos_desarrollo,
    cv=cv_grupos,
    scoring={
        "MAE": "neg_mean_absolute_error",
        "RMSE": "neg_root_mean_squared_error",
        "R2": "r2"
    }
)

modelo_base.fit(X_desarrollo, y_desarrollo)
y_pred_base = modelo_base.predict(X_prueba)
mae_base = mean_absolute_error(y_prueba, y_pred_base)
rmse_base = np.sqrt(mean_squared_error(y_prueba, y_pred_base))
r2_base = r2_score(y_prueba, y_pred_base)

print("BASELINE - mediana")
print("MAE prueba:", round(mae_base, 2))
print("RMSE prueba:", round(rmse_base, 2))
print("R? prueba:", round(r2_base, 4))
print("Mejora de MAE frente al baseline:", round(mae_base - mae_final, 2))


### Interpretaci?n de las m?tricas y errores

El MAE representa el error absoluto promedio en las unidades monetarias de `price`. El RMSE penaliza con mayor fuerza los errores grandes; por eso el diagn?stico anterior reporta el m?ximo error, la raz?n RMSE/MAE, la contribuci?n de los casos extremos al SSE y las m?tricas por cuartil de precio. R? resume la variabilidad explicada, pero no sustituye el an?lisis de errores individuales.

La comparaci?n de modelos se realiz? exclusivamente con validaci?n cruzada sobre desarrollo y por grupos de direcci?n-ciudad. La m?trica reportada sobre la prueba final corresponde a una ?nica evaluaci?n sobre grupos no vistos.


## 10. Almacenamiento y reutilización del modelo

El pipeline completo se guarda con `joblib`. Posteriormente se carga nuevamente y se utiliza para generar una predicción, demostrando que el modelo almacenado puede reutilizarse sin volver a entrenarlo.

In [ ]:
joblib.dump(modelo_final, "modelo.joblib")

modelo_cargado = joblib.load("modelo.joblib")
ejemplo = X_prueba.iloc[[0]]
prediccion = modelo_cargado.predict(ejemplo)

print("Modelo guardado y cargado correctamente.")
print("Precio real:", round(float(y_prueba.iloc[0]), 2))
print("Precio predicho:", round(float(prediccion[0]), 2))


## 11. Conclusiones

El EDA combina gr?ficos con hallazgos cuantitativos: identifica la asimetr?a del precio, la relaci?n aproximada de 0,88 entre `sqft_living` y `sqft_above`, la presencia de valores extremos y la ausencia de faltantes originales. Las variables de superficie se conservan porque representan componentes distintos de la vivienda y la decisi?n se valida mediante el desempe?o del modelo.

Los faltantes se simularon al 1 % ?nicamente como ejercicio reproducible de imputaci?n; no forman parte del dataset original. La fuente y el contexto del conjunto quedan documentados al inicio del notebook.

El an?lisis de errores examina los casos que m?s contribuyen al SSE y compara MAE/RMSE por cuartiles de precio. Esto permite explicar si el RMSE elevado est? dominado por pocas viviendas de precio extremo, en lugar de presentarlo solamente como una m?trica global.
